# CILE MD handoff - Colab Linux validation

This notebook validates a clean extraction, checksums, static inputs, and EM-stage `grompp` for all five official systems. It does not prove physical equilibration or the professor's server path.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
assert len(zip_names) == 1, f'Upload exactly one handoff ZIP, got: {zip_names}'
zip_name = zip_names[0]
print('Uploaded:', zip_name)

In [ ]:
import hashlib, pathlib, shutil, subprocess
workspace = pathlib.Path.cwd()
work = workspace / 'cile_validation'
if work.exists():
    shutil.rmtree(work)
work.mkdir()
digest = hashlib.sha256(pathlib.Path(zip_name).read_bytes()).hexdigest()
print('ZIP SHA256:', digest)
subprocess.run(['unzip', '-q', zip_name, '-d', str(work)], check=True)
packages = list(work.glob('CILE_MD_Handoff*'))
assert len(packages) == 1, f'Expected one package root, got: {packages}'
package = packages[0]
print('Package root:', package)

In [ ]:
subprocess.run(['bash', '-lc', f"cd '{package}' && sha256sum -c SHA256SUMS"], check=True)
subprocess.run(['bash', str(package / 'validate_inputs.sh')], check=True)

In [ ]:
subprocess.run(['bash', '-lc', 'apt-get -qq update && apt-get -qq install -y gromacs'], check=True)
version = subprocess.run(['gmx', '--version'], text=True, capture_output=True, check=True)
print(version.stdout)

In [ ]:
labels = ['L1P1', 'L1P2', 'L2P1', 'L3P1', 'L1P3']
out = workspace / 'cile_grompp'
out.mkdir(exist_ok=True)
results = []
for label in labels:
    label_out = out / label
    label_out.mkdir(exist_ok=True)
    system = package / 'systems' / label
    cmd = [
        'gmx', 'grompp',
        '-f', str(package / 'mdp' / '01_em_strict.mdp'),
        '-c', str(system / 'initial.gro'),
        '-p', str(system / 'topol.top'),
        '-o', str(label_out / 'em.tpr'),
        '-po', str(label_out / 'em_mdout.mdp'),
    ]
    run = subprocess.run(cmd, cwd=system, text=True, capture_output=True)
    (label_out / 'grompp.stdout.txt').write_text(run.stdout)
    (label_out / 'grompp.stderr.txt').write_text(run.stderr)
    results.append((label, run.returncode, (label_out / 'em.tpr').exists()))
print(results)
assert all(code == 0 and exists for _, code, exists in results), results

In [ ]:
import json, platform, datetime
report = {
    'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'platform': platform.platform(),
    'zip_sha256': digest,
    'gromacs_version_output': version.stdout,
    'grompp_results': results,
    'scope': 'Clean extract, checksums, static validation, and EM grompp only',
    'not_verified': ['mdrun', 'physical equilibration', 'professor lab server'],
}
(out / 'COLAB_REPORT.json').write_text(json.dumps(report, indent=2))
archive_base = workspace / 'colab_validation_artifacts'
archive_path = shutil.make_archive(str(archive_base), 'zip', out)
files.download(archive_path)